**Feature Engineering**

6) Feature Stage: Full Validation on optimized_df

In [ ]:
target_col = "nova_group"

# Except target: Auto-detect numeric features
feature_cols = [
    f.name for f in optimized_df.schema.fields
    if isinstance(f.dataType, NumericType) and f.name != target_col
]

print("Detected feature columns:", feature_cols)

total_rows = optimized_df.count()

# Null % per selected column
null_stats = []
for c in [target_col] + feature_cols:
    null_count = optimized_df.filter(F.col(c).isNull()).count()
    null_stats.append((c, null_count, (null_count / total_rows) * 100))

null_df = spark.createDataFrame(null_stats, ["column", "null_count", "null_percent"])
print("Null counts:")
null_df.show()

# Sanity checks: negatives + extremely large values
sanity_rows = []
for c in feature_cols:
    neg_count = optimized_df.filter(F.col(c) < 0).count()
    big_count = optimized_df.filter(F.col(c) > 10000).count()
    sanity_rows.append((c, neg_count, big_count))

sanity_df = spark.createDataFrame(sanity_rows, ["column", "negative_count", "too_large_count"])
print("Sanity checks:")
sanity_df.show()

# Min/Max summary
summary_exprs = []
for c in feature_cols:
    summary_exprs += [F.min(c).alias(f"{c}_min"), F.max(c).alias(f"{c}_max")]

minmax_row = optimized_df.select(*summary_exprs).collect()[0].asDict()
minmax_long = []
for c in feature_cols:
    min_val = minmax_row.get(f"{c}_min")
    max_val = minmax_row.get(f"{c}_max")
    minmax_long.append((c, float(min_val) if min_val is not None else None,
                           float(max_val) if max_val is not None else None))
minmax_df = spark.createDataFrame(minmax_long, ["column", "min_value", "max_value"])

# Duplicates
dup_count = None
if "code" in optimized_df.columns:
    dup_count = optimized_df.count() - optimized_df.select("code").distinct().count()

print("Total rows:", total_rows)
print("Duplicate count (code):", dup_count)

# Export for Tableau Dashboard 1
null_df.coalesce(1).write.mode("overwrite").option("header", True).csv("tableau_null_summary.csv")
sanity_df.coalesce(1).write.mode("overwrite").option("header", True).csv("tableau_sanity_checks.csv")
minmax_df.coalesce(1).write.mode("overwrite").option("header", True).csv("tableau_minmax_summary.csv")


7) Skip full parquet rewrite

In [ ]:
# Colab-safe: treating optimized_df as curated dataset
curated_df = optimized_df
print("Curated dataset ready (sampled):")
curated_df.limit(3).show(truncate=False)

8) Domain Specific Custom Transformer

In [ ]:
# Import required classes to define a custom Spark ML transformer
from pyspark.ml.param.shared import Param, Params
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
from pyspark.ml import Transformer
import pyspark.sql.functions as F


# Custom transformer to create a new nutrition-based feature
# This adds a weighted nutrient score using sugar, salt and fat content.
class NutrientScoreTransformer(Transformer, DefaultParamsReadable, DefaultParamsWritable):

    def __init__(self, outputCol="nutrient_score"):
        super().__init__()
        # Define output column parameter
        self.outputCol = Param(self, "outputCol", "Output column name")
        self._setDefault(outputCol=outputCol)
        self._set(outputCol=outputCol)

    # This method runs when the transformer is applied in the pipeline
    # It creates a new column calculated from existing nutritional values
    def _transform(self, dataset):
        out = self.getOrDefault(self.outputCol)
        return dataset.withColumn(
            out,
            (F.col("sugars_100g") * 0.4) +
            (F.col("salt_100g") * 0.3) +
            (F.col("fat_100g") * 0.3)
        )


# Initialise the custom transformer
cust_trans = NutrientScoreTransformer(outputCol="nutrient_score")

9) Train/Test split + checkpoint (lineage cutting)

In [ ]:
# stratification in Spark is limited so, let's keep a fixed seed and show class distributions.
train_df, test_df = curated_df.randomSplit([0.8, 0.2], seed=42)

# Cut lineage before expensive model training
train_df_ckpt = train_df.checkpoint(eager=True)
test_df_ckpt = test_df.checkpoint(eager=True)

print("Train rows:", train_df_ckpt.count(), "Test rows:", test_df_ckpt.count())

# Export class distributions per split
train_dist = train_df_ckpt.groupBy("nova_group").count().orderBy("nova_group")
test_dist = test_df_ckpt.groupBy("nova_group").count().orderBy("nova_group")
train_dist.coalesce(1).write.mode("overwrite").option("header", True).csv("class_distribution_train.csv")
test_dist.coalesce(1).write.mode("overwrite").option("header", True).csv("class_distribution_test.csv")


10) Common ML Pipeline (index → features → scaler → PCA)

In [ ]:
indexer = StringIndexer(inputCol="nova_group", outputCol="label", handleInvalid="skip")

# Use detected numeric features + engineered score
# Ensure features exist in curated_df:
base_features = [c for c in feature_cols if c != "label"]  # from earlier auto-detect

# Add engineered feature produced by transformer
assembler = VectorAssembler(inputCols=base_features + ["nutrient_score"], outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled", withMean=True, withStd=True)
pca = PCA(k=5, inputCol="features_scaled", outputCol="features")

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
